The Lite build clusters the lung surfaces at 0.12 mm for whole-organ navigation and clipping. It omits the independently measured E02 ROI volume, which is retained in Full. These are display approximations, not new anatomy or biological measurements.


In [1]:
from pathlib import Path
import base64, gzip, json, re
import numpy as np

ROOT = Path.cwd()
SOURCE = ROOT / 'viewer/mlung3d_v3.3_full.html'
OUTPUT = ROOT / 'viewer/mlung3d_v3.3_lite.html'
GRID_MM = 0.12
source_html = SOURCE.read_text()
# Remove offline-only engine/body payload from the public preview when reading Full.
if 'const {THREE,OrbitControls,OBJLoader}=MlungEngine;' in source_html:
    engine=(ROOT/'assets/three-offline-v0.180.0.js').read_text().replace('</script','<\\/script')
    source_html=source_html.replace('<script>'+engine+'</script>', '<script type="importmap">{"imports":{"three":"https://cdn.jsdelivr.net/npm/three@0.180.0/build/three.module.js","three/addons/":"https://cdn.jsdelivr.net/npm/three@0.180.0/examples/jsm/"}}</script>',1)
    source_html=source_html.replace('const {THREE,OrbitControls,OBJLoader}=MlungEngine;', "const THREE=await import('three'); const {OrbitControls}=await import('three/addons/controls/OrbitControls.js'); const {OBJLoader}=await import('three/addons/loaders/OBJLoader.js');",1)
    source_html=re.sub(r'  const mouseObjText=.*?;\n', "  const mouseObjUrl='https://raw.githubusercontent.com/anl13/MAMMAL_mouse/main/mouse_model/mouse_reduced_face_7200.obj';\n",source_html,count=1)
    source_html=source_html.replace('const text=mouseObjText;', "const text=await fetch(mouseObjUrl).then(r=>{if(!r.ok)throw new Error('mouse body download failed');return r.text()});",1)
match = re.search(r'(<script id="meshPayload" type="application/json">)(.*?)(</script>)', source_html, re.S)
assert match, 'M07 mesh payload not found'
meshes = json.loads(match.group(2))
assert len(meshes) == 6
print(f'Source: {SOURCE.name}; {len(meshes)} meshes; {SOURCE.stat().st_size/2**20:.1f} MiB')


Source: mlung3d_v3.3_full.html; 6 meshes; 44.8 MiB


## Cluster and reindex each surface

Vertices are grouped in 0.08 mm cubic cells in the unchanged M07 coordinate frame. Each cluster is represented by its mean position. Degenerate and duplicate triangles are removed; orientation is retained from the source face. This is a visual level of detail, **not an anatomical remeshing or segmentation**.


In [2]:
def decode_array(text, dtype, columns):
    return np.frombuffer(gzip.decompress(base64.b64decode(text)), dtype=dtype).reshape(-1, columns)

def encode_array(a):
    return base64.b64encode(gzip.compress(a.tobytes(order='C'), compresslevel=6, mtime=0)).decode('ascii')

def clustered_mesh(d, grid_mm):
    q = decode_array(d['q_b64'], '<u2', 3)
    faces = decode_array(d['i_b64'], '<u4', 3)
    lo, hi = np.asarray(d['min'], dtype=np.float64), np.asarray(d['max'], dtype=np.float64)
    pos = lo + q.astype(np.float64) * ((hi-lo)/65535.0)
    cells = np.floor(pos/grid_mm).astype(np.int32)
    _, inverse = np.unique(cells, axis=0, return_inverse=True)
    count = np.bincount(inverse)
    clustered = np.column_stack([np.bincount(inverse, weights=pos[:,axis])/count for axis in range(3)])
    mapped = inverse[faces]
    valid = (mapped[:,0] != mapped[:,1]) & (mapped[:,0] != mapped[:,2]) & (mapped[:,1] != mapped[:,2])
    mapped = mapped[valid]
    _, first = np.unique(np.sort(mapped, axis=1), axis=0, return_index=True)
    mapped = mapped[np.sort(first)]
    used, remapped = np.unique(mapped, return_inverse=True)
    clustered = clustered[used]
    mapped = remapped.reshape(-1,3).astype('<u4')
    qlite = np.clip(np.rint((clustered-lo)/(hi-lo)*65535),0,65535).astype('<u2')
    out = {k:d[k] for k in ('name','color','min','max')}
    out.update(verts=len(qlite), faces=len(mapped), q_b64=encode_array(qlite), i_b64=encode_array(mapped))
    return out

light_meshes = [clustered_mesh(d, GRID_MM) for d in meshes]
for old,new in zip(meshes,light_meshes):
    print(f"{old['name']}: {old['verts']:,} -> {new['verts']:,} vertices; {old['faces']:,} -> {new['faces']:,} faces")


Left lobe: 375,587 -> 33,513 vertices; 751,306 -> 67,087 faces
Right accessory lobe: 226,549 -> 20,048 vertices; 453,442 -> 40,306 faces
Right caudal lobe: 338,500 -> 30,335 vertices; 677,032 -> 60,760 faces
Right cranial lobe: 266,251 -> 23,496 vertices; 533,222 -> 47,269 faces
Right middle lobe: 213,127 -> 18,853 vertices; 426,626 -> 37,957 faces
Trachea and segmented airway surface: 551,640 -> 47,515 vertices; 1,103,154 -> 96,161 faces


## Publish a clearly labeled interactive build

The viewer UI and interaction code remain the same. The independent measured E02 ROI is retained unmodified. We add a visible notice and change the document title to identify the approximate web build. Capture buttons are hidden and replaced by a Full HTML download prompt.


## Lite runtime boundary

The online Lite viewer contains only clustered M07 lung geometry. The independent E02 alveolar volume, 2D section computation, and publication exports belong to Full. The version-pinned Three.js imports remain on jsDelivr, which was browser-tested. A same-origin module trial returned HTTP 200 but was blocked by the test browser; it was removed. The downloadable Full HTML is unchanged.


In [3]:
replacement = json.dumps(light_meshes, separators=(',',':'))
light_html = source_html[:match.start(2)] + replacement + source_html[match.end(2):]
# The independent E02 ROI belongs to Full. Its 11 MB encoded volume was previously
# parsed in Lite even when the ROI was never opened.
roi_literal = re.search(r'const alvData=\{[^\n]*\};', light_html)
assert roi_literal and len(roi_literal.group(0)) > 10_000_000
light_html = light_html[:roi_literal.start()] + 'const alvData=null;' + light_html[roi_literal.end():]
light_html = light_html.replace("document.getElementById('openAlveolar').addEventListener('click',openAlveolar);", "document.getElementById('openAlveolar').addEventListener('click',()=>{document.getElementById('liteExportGate').open=true;});", 1)
light_html = light_html.replace(">Alveolar chicken-wire view</button>", ">Alveolar ROI · Full only</button>", 1)
light_html = light_html.replace('<title>Mouse Lung Explorer 3.2 | M07 model · NAV anatomy</title>', '<title>Mouse Lung Explorer 3.2 Lite | M07 display approximation</title>', 1)
light_html = light_html.replace('<h1>Mouse lung explorer · v3.3</h1>', '<h1>Mouse lung explorer · v3.3 Lite</h1>', 1)
notice = '<p class="liteNotice" style="font-size:11px;line-height:1.45;margin:8px 0;color:#a8e3f1;border-left:2px solid #7bd9f3;padding-left:9px">Fast public preview · 0.08 mm clustered display mesh. Rotate and clip the 3D surface here. For an accurate 2D section and all exports, download and open the Full HTML locally.</p>'
light_html = light_html.replace('<h2>Model notes & citations</h2>', '<h2>Model notes & citations</h2>'+notice, 1)
light_html = light_html.replace("'Mouse lung explorer · v3.3'", "'Mouse lung explorer · v3.3 Lite'")
light_html = light_html.replace('function scheduleSlice(){', 'function scheduleSlice(){return;', 1)
# Keep motion controls, but route every capture request to the full downloadable build.
light_css = """<style id="liteCaptureGateStyle">
#saveLungPng,#recordLungGif,#recordLungVideo,#gifTimer,#pngSetup,#gifSetup,#videoSetup,#saveRoiPng,#recordRoiVideo,#slicePanel,#clipInset{display:none!important}
#liteExportGate{position:relative;color:#e3eff6;font:11px system-ui,sans-serif}
#liteExportGate summary{list-style:none;cursor:pointer;white-space:nowrap;border:1px solid #355668;border-radius:10px;padding:13px 17px;background:#10212b;font-size:15px;font-weight:750}
#liteExportGate summary::-webkit-details-marker{display:none}
#liteExportGate[open] summary{border-color:#7bd9f3}
#liteExportGate .liteGateBody{position:absolute;right:0;top:calc(100% + 8px);z-index:80;box-sizing:border-box;width:min(440px,calc(100vw - 24px));padding:24px;border:1px solid #477a8d;border-radius:18px;background:linear-gradient(145deg,#132733,#09141b);box-shadow:0 18px 48px #0009;font:16px/1.55 system-ui,sans-serif}
#liteExportGate .liteGateBody strong{display:block;font-size:23px;line-height:1.25;color:#f1f9fc}#liteExportGate .liteGateBody p{color:#c8dae3;margin:12px 0}#liteExportGate .liteGateBody ul{margin:16px 0;padding-left:21px;color:#e2eff5}#liteExportGate .liteGateBody li{margin:7px 0}#liteExportGate .liteGateBody small{display:block;margin-top:14px;font-size:14px;line-height:1.5;color:#aec7d3}
#liteExportGate .liteGateBody a{display:block;text-align:center;color:#071820;background:#a8e3f1;border-radius:12px;padding:14px 16px;text-decoration:none;font-size:18px;font-weight:750}
@media(max-width:760px){#liteExportGate .liteGateBody{top:auto;bottom:calc(100% + 8px)}}
</style>"""
light_html = light_html.replace('</head>', light_css+'</head>', 1)
light_gate = """<details id="liteExportGate"><summary>Publication export</summary><div class="liteGateBody"><strong>Explore more. Export your view.</strong><p>Detailed 3D, virtual 2D sections and alveolar views—with image, GIF and video export.</p><a href="mlung3d_v3.3_full.html" download="mlung3d-v3.3-full.html">Download Full HTML · about 47 MB</a><small>Open the downloaded HTML in your browser. No installation or internet needed.</small></div></details>"""
light_html = light_html.replace('  <div id="presentationToolbar" role="toolbar" aria-label="Presentation export and motion">', '  <div id="presentationToolbar" role="toolbar" aria-label="Motion and full-viewer download">'+light_gate, 1)
lite_clip_notice = """<aside id="liteClipNotice" role="status" hidden><button id="liteClipClose" type="button" aria-label="Dismiss notice">×</button><strong>Need a virtual 2D section?</strong><p>Download the Full viewer and open the HTML locally to compute a virtual section through this reference model.</p><a href="mlung3d_v3.3_full.html" download="mlung3d-v3.3-full.html">Download Full viewer</a></aside><script>document.getElementById('toggleClip').addEventListener('click',()=>{setTimeout(()=>{document.getElementById('liteClipNotice').hidden=document.getElementById('toggleClip').textContent.trim()!=='Unclip';},0);});document.getElementById('liteClipClose').addEventListener('click',()=>{document.getElementById('liteClipNotice').hidden=true;});</script>"""
light_html = light_html.replace('</body>', lite_clip_notice+'</body>', 1)
# Only one floating information panel should remain open at a time.
light_html = light_html.replace('</body>', """<script>document.getElementById('liteExportGate').addEventListener('toggle',e=>{if(e.target.open){const info=document.getElementById('infoPanel');if(info)info.hidden=true;}});document.getElementById('toggleInfo').addEventListener('click',()=>{document.getElementById('liteExportGate').open=false;});</script></body>""", 1)
light_html = light_html.replace('</style>', """#liteClipNotice{position:fixed;right:20px;top:92px;z-index:95;width:min(340px,calc(100vw - 40px));padding:18px 20px;border:1px solid #4a8398;border-radius:14px;background:#0b1a24;color:#e7f5fa;box-shadow:0 18px 40px #0009;font:14px/1.45 system-ui,sans-serif}#liteClipNotice[hidden]{display:none}#liteClipNotice strong{display:block;font-size:16px}#liteClipNotice p{margin:8px 0 14px;color:#bbd0db}#liteClipNotice a{color:#091923;background:#a8e3f1;border-radius:8px;padding:9px 12px;text-decoration:none;font-weight:750;display:inline-block}#liteClipClose{position:absolute;right:8px;top:7px;border:0;background:none;color:#b8d8e3;font-size:22px;cursor:pointer}@media(max-width:760px){#liteClipNotice{top:auto;bottom:68px;right:10px}}"""+'</style>', 1)
assert 'id="liteClipNotice"' in light_html
assert 'id="liteExportGate"' in light_html

assert 'v3.3 Lite' in light_html and len(light_html) < len(source_html)
# Do not leave the initial screen frozen if the 3D module never starts.
light_html = light_html.replace('Loading high-detail geometry…', 'Loading Lite lung model…', 1)
startup_watchdog = '''<script>setTimeout(()=>{const box=document.getElementById('status'),detail=document.getElementById('progressText');if(box&&detail&&getComputedStyle(box).display!=='none'&&detail.textContent==='Preparing 3D engine'){detail.textContent='The 3D engine is taking longer than expected.';const retry=document.createElement('button');retry.type='button';retry.textContent='Reload viewer';retry.style.marginTop='12px';retry.addEventListener('click',()=>location.reload());box.appendChild(retry);}},12000);</script>'''
light_html = light_html.replace('</body>', startup_watchdog+'</body>', 1)
welcome_modal = '<style>\n#liteWelcome{position:fixed;inset:0;z-index:1000;display:grid;place-items:center;padding:20px;background:rgba(2,7,11,.82);backdrop-filter:blur(10px)}#liteWelcome[hidden]{display:none}\n#liteWelcomeCard{box-sizing:border-box;width:min(500px,100%);padding:32px;border:1px solid #446777;border-radius:24px;background:linear-gradient(145deg,#142a36,#09141c);box-shadow:0 24px 90px #0009;color:#eaf5fa;font:16px/1.6 system-ui,sans-serif}\n#liteWelcomeCard h2{font-size:29px;line-height:1.2;margin:0 0 16px;letter-spacing:-.025em}#liteWelcomeCard p{margin:0 0 23px;color:#c0d4de}\n#liteWelcomeDownload{display:block;text-align:center;padding:15px 18px;border-radius:12px;background:#62d5f0;color:#071922;font-size:18px;font-weight:750;text-decoration:none;box-shadow:0 5px 24px #36c5e52a}\n#liteWelcomeDownload:hover{background:#92e8fa}#liteWelcomeContinue{display:block;width:100%;margin-top:14px;box-sizing:border-box;padding:13px 18px;border:1px solid #7193a3;border-radius:12px;background:transparent;color:#d6e7ef;font-size:16px;font-weight:600;cursor:pointer}#liteWelcomeContinue:hover{border-color:#b2dcec;color:#f2fbff}\n#liteWelcome a:focus-visible,#liteWelcome button:focus-visible{outline:3px solid #f0f9fc;outline-offset:4px}\n@media(max-width:520px){#liteWelcomeCard{padding:25px}#liteWelcomeCard h2{font-size:25px}}\n</style>\n<div id="liteWelcome" role="dialog" aria-modal="true" aria-labelledby="liteWelcomeTitle" hidden><section id="liteWelcomeCard"><h2 id="liteWelcomeTitle">Explore online. Download for more.</h2><p>Lite lets you rotate and clip the 3D lung online. Download Full for detailed anatomy, virtual 2D sections, alveolar views, and image or GIF/video exports. Open the downloaded HTML file in your browser — no installation or internet needed.</p><a id="liteWelcomeDownload" href="mlung3d_v3.3_full.html" download="mlung3d-v3.3-full.html">Download Full viewer · 47 MB</a><button id="liteWelcomeContinue" type="button">Explore online with Lite</button></section></div>\n<script>(()=>{const key=\'mlung3d-lite-welcome-seen\',dialog=document.getElementById(\'liteWelcome\'),download=document.getElementById(\'liteWelcomeDownload\'),skip=document.getElementById(\'liteWelcomeContinue\');let seen=false;try{seen=localStorage.getItem(key)===\'1\'}catch{}if(seen)return;dialog.hidden=false;const previous=document.activeElement;for(const id of [\'app\',\'uiResizeGrip\']){const node=document.getElementById(id);if(node)node.inert=true}download.focus();function dismiss(){try{localStorage.setItem(key,\'1\')}catch{}dialog.hidden=true;for(const id of [\'app\',\'uiResizeGrip\']){const node=document.getElementById(id);if(node)node.inert=false}if(previous&&previous.focus)previous.focus()}download.addEventListener(\'click\',dismiss);skip.addEventListener(\'click\',dismiss);dialog.addEventListener(\'keydown\',e=>{if(e.key===\'Escape\'){e.preventDefault();dismiss()}if(e.key===\'Tab\'){e.preventDefault();(document.activeElement===download?skip:download).focus()}})})();</script>'
light_html = light_html.replace('</body>', welcome_modal+'</body>', 1)
OUTPUT.write_text(light_html)
report = {'source':str(SOURCE.relative_to(ROOT)),'output':str(OUTPUT.relative_to(ROOT)),'grid_mm':GRID_MM,'source_bytes':SOURCE.stat().st_size,'light_bytes':OUTPUT.stat().st_size,'source_vertices':sum(d['verts'] for d in meshes),'light_vertices':sum(d['verts'] for d in light_meshes),'source_faces':sum(d['faces'] for d in meshes),'light_faces':sum(d['faces'] for d in light_meshes),'purpose':'display-only approximation; 3D clip allowed, E02 ROI/2D section/exports delegated to Full'}
(ROOT/'qa/public_light_viewer_report.json').write_text(json.dumps(report,indent=2)+'\n')
print(f"Light HTML: {OUTPUT.stat().st_size/2**20:.1f} MiB ({100*OUTPUT.stat().st_size/SOURCE.stat().st_size:.1f}% of full)")


Light HTML: 4.0 MiB (8.9% of full)


## Validation

Check the generated HTML in a browser: visible lite label; mouse-lung model rendering; rotation and plane movement; functional 3D Clip with 2D section hidden; no visible capture controls; Record / export opens a full-viewer download message; no console errors. Compare screenshots at the same plane with the full viewer. This is a display-quality check only.
